# 6. Multi-Seed Sweep — one cell per run (12 total)

Run setup cells 1–3 first, then run cells 4–15 in order.

**GPT** identity / W&B id: `tinystories_large_{arch}_ctx512_steps3000_seed{SEED}`
**VLM** run name: `vlm_{arch}_flickr30k_b8_seed{SEED}`

Seed `42` is excluded (already finished). Each run cell skips automatically if outputs already exist.

On Colab, **cell 3** mounts Google Drive and writes checkpoints to `MyDrive/AttnResGPT-next-scale-artifacts/` (survives runtime disconnect).


## 1. Sync repo and install deps

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-next-scale.git'


def is_repo_root(path: Path) -> bool:
    return (path / 'src' / 'training' / 'train.py').exists() and (path / 'requirements.txt').exists()


def discover_repo() -> Path | None:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()
    for root in [Path('/content'), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        for path in root.rglob('AttnResGPT-next-scale'):
            if is_repo_root(path):
                return path.resolve()
    return None


REPO_DIR = discover_repo()
if REPO_DIR is None:
    target = Path('/content/AttnResGPT-next-scale')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    REPO_DIR = target.resolve()

os.chdir(REPO_DIR)
print(f'Using repo at: {REPO_DIR}')
subprocess.run(['git', 'fetch', '--quiet'], check=False)
subprocess.run(['git', 'pull', '--ff-only'], check=False)
subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

Using repo at: /content/AttnResGPT-next-scale


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 2. W&B + GPU

In [ ]:
import os
import torch

# Set WANDB_API_KEY in Colab secrets or uncomment:
os.environ.setdefault('WANDB_API_KEY', '')  # set via Colab secrets or uncomment below
# os.environ['WANDB_API_KEY'] = '...'
print('WANDB_API_KEY set =', bool(os.environ.get('WANDB_API_KEY')))
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Switch Colab runtime to GPU before launching sweeps.')

WANDB_API_KEY set = False
cuda_available = True
device_name = Tesla T4


## 3. Launch helpers (used by run cells below)

In [3]:
import shutil
import subprocess
import sys
from pathlib import Path

SKIP_EXISTING = True
GPT_CONFIG = 'configs/large_ctx512_3000.yaml'
DRIVE_ARTIFACT_FOLDER = 'AttnResGPT-next-scale-artifacts'
USE_GOOGLE_DRIVE = True  # False = keep artifacts on the Colab VM (ephemeral)
VLM_BATCH_SIZE = 8
VLM_NUM_WORKERS = 2
VLM_FINAL_CHECKPOINT = 'step_0006750.pt'


def repo_root() -> Path:
    """Paths are relative to repo root (cell 1 runs os.chdir(REPO_DIR))."""
    root = Path.cwd().resolve()
    if (root / 'src' / 'training' / 'train.py').exists():
        return root
    fallback = Path('/content/AttnResGPT-next-scale').resolve()
    if (fallback / 'src' / 'training' / 'train.py').exists():
        return fallback
    raise RuntimeError('Run cell 1 first — cannot find repo root.')


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _symlink_to_drive(repo: Path, name: str, target: Path) -> None:
    local = repo / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == target.resolve():
            return
        local.unlink()
    elif local.exists():
        if local.is_file():
            local.unlink()
        else:
            for child in local.iterdir():
                if child.name.startswith('.'):
                    if child.is_dir():
                        shutil.rmtree(child)
                    else:
                        child.unlink()
                    continue
                dest = target / child.name
                if dest.exists():
                    shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
                shutil.move(str(child), str(dest))
            shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)
    print(f'  linked {name} -> {target}')


def setup_artifact_storage(*, use_drive: bool = USE_GOOGLE_DRIVE) -> Path:
    """Return GPT logging.output_root; symlink VLM checkpoints/outputs to Drive on Colab."""
    repo = repo_root()
    local_root = (repo / 'outputs').resolve()
    if not use_drive or not _in_colab():
        print('Artifacts on VM disk:', local_root)
        return local_root

    from google.colab import drive

    drive.mount('/content/drive')
    drive_root = (Path('/content/drive/MyDrive') / DRIVE_ARTIFACT_FOLDER).resolve()
    for sub in ('runs', 'checkpoints', 'logs', 'outputs'):
        (drive_root / sub).mkdir(parents=True, exist_ok=True)

    print('Artifacts on Google Drive:', drive_root)
    _symlink_to_drive(repo, 'checkpoints', drive_root / 'checkpoints')
    _symlink_to_drive(repo, 'outputs', drive_root / 'outputs')
    return drive_root


OUTPUT_ROOT = setup_artifact_storage()


def gpt_identity(architecture: str, seed: int) -> str:
    return f'tinystories_large_{architecture}_ctx512_steps3000_seed{seed}'


def gpt_paths(architecture: str, seed: int, *, root: Path | None = None) -> dict:
    root = root or repo_root()
    identity = gpt_identity(architecture, seed)
    run_dir = OUTPUT_ROOT / 'runs' / identity
    return {
        'root': root,
        'identity': identity,
        'run_dir': run_dir,
        'checkpoint_dir': OUTPUT_ROOT / 'checkpoints' / identity,
        'summary': run_dir / 'run_summary.json',
        'blockers': [
            run_dir / 'train_metrics.jsonl',
            run_dir / 'val_metrics.jsonl',
            run_dir / 'run_summary.json',
        ],
        'global_logs': sorted((OUTPUT_ROOT / 'logs').glob(f'{identity}_*.jsonl')),
    }


def reset_gpt_run(architecture: str, seed: int) -> None:
    """Delete partial/failed run outputs (same dirs train.py uses)."""
    paths = gpt_paths(architecture, seed)
    print(f'repo root: {paths["root"]}')
    print(f'resetting: {paths["identity"]}')
    for label in ('run_dir', 'checkpoint_dir'):
        path = paths[label]
        if path.exists():
            shutil.rmtree(path)
            print(f'  removed {label}: {path}')
    for path in paths['global_logs']:
        path.unlink(missing_ok=True)
        print(f'  removed log: {path}')
    remaining = [p for p in paths['blockers'] if p.exists()]
    if remaining:
        raise RuntimeError(f'cleanup incomplete: {remaining}')
    print('  ok — safe to launch')


def launch_gpt(architecture: str, seed: int) -> None:
    paths = gpt_paths(architecture, seed)
    identity = paths['identity']
    root = paths['root']
    if SKIP_EXISTING and paths['summary'].exists():
        print(f'SKIP (complete): {identity}')
        return
    if any(p.exists() for p in paths['blockers']):
        print(f'partial outputs for {identity} — clearing before launch')
        reset_gpt_run(architecture, seed)
    cmd = [
        sys.executable, '-m', 'src.training.train',
        '--config', GPT_CONFIG,
        '--model', architecture,
        '--overrides',
        f'experiment.seed={seed}',
        f'logging.output_root={OUTPUT_ROOT}',
        f'logging.wandb.tags=[multiseed,large,ctx512,{architecture},seed{seed}]',
    ]
    print(f'Launching {identity} from {root}')
    subprocess.run(cmd, check=True, cwd=root)


def vlm_run_name(architecture: str, seed: int) -> str:
    return f'vlm_{architecture}_flickr30k_b{VLM_BATCH_SIZE}_seed{seed}'


def launch_vlm(architecture: str, seed: int) -> None:
    root = repo_root()
    run_name = vlm_run_name(architecture, seed)
    ckpt = root / 'checkpoints' / run_name / VLM_FINAL_CHECKPOINT
    if SKIP_EXISTING and ckpt.exists():
        print(f'SKIP (exists): {run_name}')
        return
    cmd = [
        sys.executable, 'experiments/vlm_attnres_flickr30k.py',
        '--decoder-architecture', architecture,
        '--seed', str(seed),
        '--batch-size', str(VLM_BATCH_SIZE),
        '--num-workers', str(VLM_NUM_WORKERS),
        '--run-name', run_name,
    ]
    print(f'Launching {run_name} from {root}')
    subprocess.run(cmd, check=True, cwd=root)

Mounted at /content/drive
Artifacts on Google Drive: /content/drive/MyDrive/AttnResGPT-next-scale-artifacts
  linked checkpoints -> /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/checkpoints
  linked outputs -> /content/drive/MyDrive/AttnResGPT-next-scale-artifacts/outputs


### Run 1/12 — GPT baseline, seed 123
`tinystories_large_baseline_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('baseline', 123)

### Run 2/12 — GPT attnres, seed 123
`tinystories_large_attnres_ctx512_steps3000_seed123`

In [ ]:
launch_gpt('attnres', 123)

### Run 3/12 — GPT baseline, seed 456
`tinystories_large_baseline_ctx512_steps3000_seed456`

In [ ]:
launch_gpt('baseline', 456)

### Run 4/12 — GPT attnres, seed 456
`tinystories_large_attnres_ctx512_steps3000_seed456`

In [ ]:
launch_gpt('attnres', 456)

### Run 5/12 — GPT baseline, seed 789
`tinystories_large_baseline_ctx512_steps3000_seed789`

In [ ]:
launch_gpt('baseline', 789)

### Run 6/12 — GPT attnres, seed 789
`tinystories_large_attnres_ctx512_steps3000_seed789`

In [ ]:
launch_gpt('attnres', 789)

### Run 7/12 — VLM baseline, seed 123
`vlm_baseline_flickr30k_b8_seed123`

In [4]:
launch_vlm('baseline', 123)

Launching vlm_baseline_flickr30k_b8_seed123 from /content/AttnResGPT-next-scale


CalledProcessError: Command '['/usr/bin/python3', 'experiments/vlm_attnres_flickr30k.py', '--decoder-architecture', 'baseline', '--seed', '123', '--batch-size', '8', '--num-workers', '2', '--run-name', 'vlm_baseline_flickr30k_b8_seed123']' returned non-zero exit status 1.

### Run 8/12 — VLM attnres, seed 123
`vlm_attnres_flickr30k_b8_seed123`

In [12]:
launch_vlm('attnres', 123)

### Run 9/12 — VLM baseline, seed 456
`vlm_baseline_flickr30k_b8_seed456`

In [13]:
launch_vlm('baseline', 456)

Launching vlm_baseline_flickr30k_b8_seed456 from /content/AttnResGPT-next-scale


### Run 10/12 — VLM attnres, seed 456
`vlm_attnres_flickr30k_b8_seed456`

In [14]:
launch_vlm('attnres', 456)

Launching vlm_attnres_flickr30k_b8_seed456 from /content/AttnResGPT-next-scale


### Run 11/12 — VLM baseline, seed 789
`vlm_baseline_flickr30k_b8_seed789`

In [15]:
launch_vlm('baseline', 789)

Launching vlm_baseline_flickr30k_b8_seed789 from /content/AttnResGPT-next-scale


### Run 12/12 — VLM attnres, seed 789
`vlm_attnres_flickr30k_b8_seed789`

In [16]:
launch_vlm('attnres', 789)

Launching vlm_attnres_flickr30k_b8_seed789 from /content/AttnResGPT-next-scale


## Post-sweep sanity check

In [ ]:
import json
from pathlib import Path

SEEDS = [123, 456, 789]
ARCHITECTURES = ['baseline', 'attnres']

print('=== GPT summaries ===')
for architecture in ARCHITECTURES:
    for seed in SEEDS:
        identity = gpt_identity(architecture, seed)
        summary_path = OUTPUT_ROOT / 'runs' / identity / 'run_summary.json'
        if not summary_path.exists():
            print(f'MISSING {identity}')
            continue
        summary = json.loads(summary_path.read_text())
        print(
            f"{identity}: val_loss={summary.get('val_loss'):.4f} "
            f"ppl={summary.get('perplexity'):.2f}"
        )

print('\n=== VLM checkpoints ===')
for architecture in ARCHITECTURES:
    for seed in SEEDS:
        run_name = vlm_run_name(architecture, seed)
        ckpt = repo_root() / 'checkpoints' / run_name / VLM_FINAL_CHECKPOINT
        status = 'OK' if ckpt.exists() else 'MISSING'
        print(f'{run_name}: {status}')


=== GPT summaries ===
MISSING tinystories_large_baseline_ctx512_steps3000_seed123
MISSING tinystories_large_baseline_ctx512_steps3000_seed456
MISSING tinystories_large_baseline_ctx512_steps3000_seed789
MISSING tinystories_large_attnres_ctx512_steps3000_seed123
MISSING tinystories_large_attnres_ctx512_steps3000_seed456
MISSING tinystories_large_attnres_ctx512_steps3000_seed789

=== VLM checkpoints ===
vlm_baseline_flickr30k_b8_seed123: MISSING
vlm_baseline_flickr30k_b8_seed456: OK
vlm_baseline_flickr30k_b8_seed789: OK
vlm_attnres_flickr30k_b8_seed123: OK
vlm_attnres_flickr30k_b8_seed456: OK
vlm_attnres_flickr30k_b8_seed789: OK
